In [ ]:
!pip install captum



In [11]:
import numpy as np
import pandas as pd
import torch
dt = torch.randn(1000, 65).float()
data = pd.DataFrame(dt)
df = data.copy() # For keep the main dataset, we will do a copy of it


In [15]:
def dataset_cleaning(data):
  df = data.copy()
  #First, we will look for the column containing the date
  date_column = None
  for col in df.columns: # We will search for the column containing dates, while requiring the user to place this column first to avoid unnecessary loops
    converted = pd.to_datetime(df[col], errors='coerce')
    if converted.notna().sum() > len(df) * 0.5:
      df[col] = converted
      date_column = col
      break

  #To ensure there are no absurd numbers that could ruin the code, we'll check if there are any and, if so, remove them
  for col in df.columns:
    if col == date_column:
      continue
    converted = pd.to_numeric(df[col], errors='coerce')
    if converted.isna().sum() > df[col].isna().sum():
      df[col] = pd.to_numeric(
          df[col].astype(str).str.replace(r'[^0-9.-]', '', regex=True),
          errors='coerce',
    )
    else:
      df[col] = converted

  sorting = df[date_column].is_monotonic_increasing # We will first check if the data is in order
  if not sorting:
    df = df.sort_values(by= date_column).reset_index(drop=True) # We will put the dates in order for better readability

  # We will use Periodic Feature Encoding so that the model can understand
  month = df[date_column].dt.month
  df['month_sin'] = np.sin(2 * np.pi * month / 12)
  df['month_cos'] = np.cos(2 * np.pi * month / 12)

  if df.isna().sum().sum() > 0 : # We apply a condition to clean the dataset if there are NaN values
    df = df.interpolate(method='linear').bfill().ffill()

  return df



In [16]:
date = pd.date_range(start="2026-01-01", periods=100, freq="D")
data = pd.DataFrame(
    {
        "date": date,
        "valeur_1": np.random.randn(100) * 10 + 50,
        "valeur_2": np.random.randn(100) * 5 + 20,
    }
)
df = dataset_cleaning(data)

In [13]:
def parameter_slide_windows(): # We will create a function to query N and K so that the code does not return a sequence of errors
  while True:
    try:
      N = int(input('Enter the size of the history window '))
      K = int(input('Enter the prediction horizon '))
      if K > 0 and N > 0:
        return N, K
      else:
        print("Please enter positive integers")
    except ValueError:
      print("Please enter valid integers")

N, K = parameter_slide_windows()

# To be ensure that N + K <= len(data) we'll create a condition for make sure that if N + K > len(data) the code will still working
if len(df) < (N + K):
  new_rows = []
  while len(df) + len(new_rows) < (N + K):
    # To add rows to the dataset in a coherent manner, we'll use the Additive White Gaussian Noise ( AWGN ) methods
    rows = np.random.normal(
        loc= df.mean(numeric_only= True),
        scale= df.std(numeric_only= True) * 0.05 # to achieve a slight oscillation for our LSTM
    )
    new_rows.append(rows)

  new_df = pd.DataFrame(new_rows, columns= df.select_dtypes(include=[np.number]).columns)
  df = pd.concat([df, new_df], ignore_index= True)

# Now, to have our X and y, we will use the sliding window technique
def sliding_window():
  X, y = [], []
  for i in range(len(df) - N - K + 1 ): # N days have passed and K days to predict
    X_window = df[i : i + N]
    y_window = df[i + N : i + N + K]

    X.append(X_window)
    y.append(y_window)

  return np.array(X), np.array(y)

Enter the size of the history window2
Enter the prediction horizon3
